# Notebook 16: Hybrid Quantum-Classical Models

---

## Overview

Hybrid quantum-classical models represent the most practical approach to leveraging quantum computing in the **Noisy Intermediate-Scale Quantum (NISQ)** era. These architectures combine classical neural networks with parameterized quantum circuits, enabling us to harness quantum computational advantages while mitigating the limitations of current quantum hardware.

### Learning Objectives

1. Understand the hybrid quantum-classical paradigm and why it matters
2. Learn how variational quantum circuits serve as differentiable ML layers
3. Master the **parameter shift rule** for computing quantum gradients
4. Explore transfer learning with quantum circuits
5. Build and train a hybrid PyTorch + PennyLane model on MNIST

### Prerequisites
- Notebooks 14-15 (QML intro, kernel methods)
- Familiarity with PyTorch and basic neural networks
- Understanding of variational quantum circuits (Notebook 10)

In [ ]:
# Installation and imports
# !pip install pennylane pennylane-qiskit torch torchvision matplotlib numpy scikit-learn

import pennylane as qml
from pennylane import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PennyLane version: {qml.__version__}")
print(f"PyTorch version: {torch.__version__}")

## 1. The Hybrid Quantum-Classical Paradigm

### Why Hybrid?

Pure quantum algorithms (Shor's, Grover's) require fault-tolerant quantum computers with thousands of logical qubits. Current NISQ devices have:
- **Limited qubits** (50-1000 noisy physical qubits)
- **Short coherence times** (microseconds to milliseconds)
- **High gate error rates** ($10^{-3}$ to $10^{-2}$)

The hybrid approach addresses this by:

$$\text{Hybrid Model} = \underbrace{f_{\text{classical}}(\mathbf{x}; \boldsymbol{\phi})}_{\text{Classical NN}} \circ \underbrace{U(\boldsymbol{\theta})}_{\text{Quantum Circuit}} \circ \underbrace{g_{\text{classical}}(\cdot; \boldsymbol{\psi})}_{\text{Classical Post-processing}}$$

### Architecture Pattern

The general workflow is:

1. **Classical Pre-processing**: Reduce high-dimensional input $\mathbf{x} \in \mathbb{R}^d$ to a manageable size $\mathbf{z} \in \mathbb{R}^n$ (where $n$ = number of qubits)
2. **Quantum Encoding**: Embed $\mathbf{z}$ into a quantum state $|\psi(\mathbf{z})\rangle$
3. **Variational Circuit**: Apply parameterized unitary $U(\boldsymbol{\theta})$
4. **Measurement**: Extract classical information via expectation values
5. **Classical Post-processing**: Map measurements to predictions

### Mathematical Framework

Given input $\mathbf{x}$, the hybrid model computes:

$$\hat{y} = h\left(\langle 0 | U^\dagger(\boldsymbol{\theta}) \, S^\dagger(\mathbf{x}) \, \hat{O} \, S(\mathbf{x}) \, U(\boldsymbol{\theta}) | 0 \rangle\right)$$

where:
- $S(\mathbf{x})$ is the **data encoding** unitary
- $U(\boldsymbol{\theta})$ is the **variational ansatz**
- $\hat{O}$ is the **measurement observable**
- $h(\cdot)$ is the **classical post-processing** function

In [ ]:
# Simple demonstration of a hybrid quantum-classical computation

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def simple_hybrid_circuit(inputs, weights):
    """A simple hybrid circuit: encode classical data, apply variational layers, measure."""
    # Data encoding layer
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)
    
    # Variational layer 1
    for i in range(n_qubits):
        qml.RY(weights[0, i], wires=i)
        qml.RZ(weights[1, i], wires=i)
    
    # Entangling layer
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])
    qml.CNOT(wires=[n_qubits - 1, 0])  # Circular entanglement
    
    # Variational layer 2
    for i in range(n_qubits):
        qml.RY(weights[2, i], wires=i)
        qml.RZ(weights[3, i], wires=i)
    
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Test with random data
inputs = np.random.uniform(0, np.pi, n_qubits)
weights = np.random.uniform(0, 2 * np.pi, (4, n_qubits))

result = simple_hybrid_circuit(inputs, weights)
print(f"Input features: {inputs}")
print(f"Circuit output (expectation values): {result}")

# Visualize the circuit
print("\nCircuit diagram:")
print(qml.draw(simple_hybrid_circuit, decimals=2)(inputs, weights))

## 2. Variational Circuits as Machine Learning Layers

### The Quantum Layer Concept

A variational quantum circuit can be viewed as a **neural network layer** that maps inputs to outputs through a parameterized unitary transformation. The key insight is that quantum circuits are **inherently differentiable** with respect to their parameters.

A single variational layer consists of:

$$\mathcal{L}_l(\boldsymbol{\theta}_l) = \prod_{i=1}^{n} R_z(\theta_{l,i}^{(2)}) R_y(\theta_{l,i}^{(1)}) \cdot \text{Entangle}(\text{wires})$$

### Expressibility of Quantum Layers

The **expressibility** of a parameterized quantum circuit measures how uniformly it can explore the Hilbert space. For a circuit generating states $|\psi(\boldsymbol{\theta})\rangle$, expressibility is quantified by:

$$\text{Expr} = D_{KL}\left(\hat{P}_{\text{PQC}}(F) \| P_{\text{Haar}}(F)\right)$$

where $F = |\langle \psi(\boldsymbol{\theta}) | \psi(\boldsymbol{\phi}) \rangle|^2$ is the **fidelity** between two random states generated by the circuit, $\hat{P}_{\text{PQC}}$ is the estimated fidelity distribution, and $P_{\text{Haar}}$ is the Haar-random fidelity distribution.

### Entangling Capability

The entangling capability measures the circuit's ability to generate entanglement:

$$\text{Ent} = \frac{1}{|S|} \sum_{\boldsymbol{\theta} \in S} Q(|\psi(\boldsymbol{\theta})\rangle)$$

where $Q$ is the **Meyer-Wallach entanglement measure**:

$$Q(|\psi\rangle) = \frac{2}{n} \sum_{k=1}^{n} \left(1 - \text{Tr}(\rho_k^2)\right)$$

with $\rho_k$ being the reduced density matrix of the $k$-th qubit.

In [ ]:
# Comparing different variational circuit architectures

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

def circuit_1(params, x):
    """Simple rotation-only circuit (low expressibility)."""
    for i in range(n_qubits):
        qml.RX(x[i], wires=i)
    for i in range(n_qubits):
        qml.RY(params[i], wires=i)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

def circuit_2(params, x):
    """Strongly entangling circuit (high expressibility)."""
    for i in range(n_qubits):
        qml.RX(x[i], wires=i)
    qml.StronglyEntanglingLayers(params, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

qnode_1 = qml.QNode(circuit_1, dev)
qnode_2 = qml.QNode(circuit_2, dev)

# Generate output distributions
n_samples = 500
x_test = np.random.uniform(0, np.pi, n_qubits)

outputs_1, outputs_2 = [], []
for _ in range(n_samples):
    p1 = np.random.uniform(0, 2 * np.pi, n_qubits)
    p2 = np.random.uniform(0, 2 * np.pi, (2, n_qubits, 3))  # 2 layers
    outputs_1.append(qnode_1(p1, x_test))
    outputs_2.append(qnode_2(p2, x_test))

outputs_1 = np.array(outputs_1)
outputs_2 = np.array(outputs_2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist2d(outputs_1[:, 0], outputs_1[:, 1], bins=30, cmap='Blues')
axes[0].set_title('Simple Circuit (Low Expressibility)', fontsize=13)
axes[0].set_xlabel('$\\langle Z_0 \\rangle$')
axes[0].set_ylabel('$\\langle Z_1 \\rangle$')

axes[1].hist2d(outputs_2[:, 0], outputs_2[:, 1], bins=30, cmap='Reds')
axes[1].set_title('Strongly Entangling Circuit (High Expressibility)', fontsize=13)
axes[1].set_xlabel('$\\langle Z_0 \\rangle$')
axes[1].set_ylabel('$\\langle Z_1 \\rangle$')

plt.suptitle('Output Distribution: Expressibility Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. The Parameter Shift Rule for Gradient Computation

### The Central Challenge

To train hybrid models using gradient-based optimization, we need to compute:

$$\frac{\partial}{\partial \theta_j} \langle \hat{O} \rangle_{\boldsymbol{\theta}} = \frac{\partial}{\partial \theta_j} \langle 0 | U^\dagger(\boldsymbol{\theta}) \hat{O} \, U(\boldsymbol{\theta}) | 0 \rangle$$

Unlike classical neural networks where we use backpropagation, quantum circuits require special techniques because:
1. We cannot directly access the quantum state (measurement collapses it)
2. The chain rule applies differently to unitary operations

### The Parameter Shift Rule

For a gate of the form $G(\theta) = e^{-i\theta \hat{G}/2}$ where $\hat{G}$ has eigenvalues $\pm 1$ (e.g., Pauli rotations), the gradient is:

$$\boxed{\frac{\partial}{\partial \theta_j} \langle \hat{O} \rangle_{\boldsymbol{\theta}} = \frac{1}{2} \left[ \langle \hat{O} \rangle_{\boldsymbol{\theta} + \frac{\pi}{2} \mathbf{e}_j} - \langle \hat{O} \rangle_{\boldsymbol{\theta} - \frac{\pi}{2} \mathbf{e}_j} \right]}$$

where $\mathbf{e}_j$ is the unit vector in the $j$-th direction.

### Derivation

Consider a circuit with a single parameterized gate $R_y(\theta) = e^{-i\theta Y/2}$. The expectation value is:

$$f(\theta) = \langle \psi | R_y^\dagger(\theta) \hat{O} R_y(\theta) | \psi \rangle$$

Since $R_y(\theta) = \cos(\theta/2) I - i \sin(\theta/2) Y$, we can write:

$$f(\theta) = A \cos(\theta) + B \sin(\theta) + C$$

for some constants $A$, $B$, $C$ that depend on the rest of the circuit. Taking the derivative:

$$f'(\theta) = -A \sin(\theta) + B \cos(\theta)$$

Now observe that:

$$f\left(\theta + \frac{\pi}{2}\right) - f\left(\theta - \frac{\pi}{2}\right) = 2(-A \sin\theta + B \cos\theta) = 2 f'(\theta)$$

Therefore:

$$f'(\theta) = \frac{f(\theta + \pi/2) - f(\theta - \pi/2)}{2}$$

This is **exact** (not a finite-difference approximation) and requires only **two circuit evaluations** per parameter!

In [ ]:
# Demonstrating the parameter shift rule

dev_grad = qml.device("default.qubit", wires=1)

@qml.qnode(dev_grad)
def param_circuit(theta):
    qml.RX(theta, wires=0)
    return qml.expval(qml.PauliZ(0))

# Analytical gradient: d/dtheta <Z> = d/dtheta cos(theta) = -sin(theta)
thetas = np.linspace(0, 2 * np.pi, 50)

# Method 1: Analytical
analytical_grads = -np.sin(thetas)

# Method 2: Parameter shift rule (manual)
shift = np.pi / 2
psr_grads = []
for theta in thetas:
    grad = (param_circuit(theta + shift) - param_circuit(theta - shift)) / 2
    psr_grads.append(grad)
psr_grads = np.array(psr_grads)

# Method 3: Finite difference
epsilon = 0.01
fd_grads = []
for theta in thetas:
    grad = (param_circuit(theta + epsilon) - param_circuit(theta - epsilon)) / (2 * epsilon)
    fd_grads.append(grad)
fd_grads = np.array(fd_grads)

# Method 4: PennyLane autograd
pennylane_grads = [qml.grad(param_circuit)(theta) for theta in thetas]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(thetas, [param_circuit(t) for t in thetas], 'b-', linewidth=2, label='$f(\\theta) = \\cos(\\theta)$')
ax1.plot(thetas, analytical_grads, 'r--', linewidth=2, label="$f'(\\theta) = -\\sin(\\theta)$")
ax1.set_xlabel('$\\theta$', fontsize=12)
ax1.set_ylabel('Value', fontsize=12)
ax1.set_title('Function and Its Gradient', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(thetas, analytical_grads, 'k-', linewidth=2, label='Analytical', alpha=0.8)
ax2.plot(thetas, psr_grads, 'r--', linewidth=2, label='Parameter Shift')
ax2.plot(thetas, fd_grads, 'g:', linewidth=2, label='Finite Difference')
ax2.plot(thetas, pennylane_grads, 'b^', markersize=4, label='PennyLane Autograd', alpha=0.6)
ax2.set_xlabel('$\\theta$', fontsize=12)
ax2.set_ylabel('Gradient', fontsize=12)
ax2.set_title('Gradient Methods Comparison', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Max error (Parameter Shift vs Analytical): {np.max(np.abs(psr_grads - analytical_grads)):.2e}")
print(f"Max error (Finite Difference vs Analytical): {np.max(np.abs(fd_grads - analytical_grads)):.2e}")

### Generalized Parameter Shift Rule

For gates with generator eigenvalues that are not limited to $\pm 1$, we use the **generalized parameter shift rule**. For a gate $G(\theta) = e^{-i\theta \hat{G}}$ with eigenvalue gaps $\{\mu_k\}$:

$$\frac{\partial f}{\partial \theta} = \sum_{k} c_k \left[ f(\theta + s_k) - f(\theta - s_k) \right]$$

where the shifts $s_k$ and coefficients $c_k$ depend on the eigenvalue spectrum.

### Higher-Order Derivatives

The parameter shift rule can be applied recursively for higher-order derivatives. The second derivative:

$$\frac{\partial^2 f}{\partial \theta^2} = \frac{1}{2}\left[f'\left(\theta + \frac{\pi}{2}\right) - f'\left(\theta - \frac{\pi}{2}\right)\right]$$

Expanding each $f'$ using the shift rule again:

$$\frac{\partial^2 f}{\partial \theta^2} = \frac{1}{4}\left[f(\theta + \pi) - 2f(\theta) + f(\theta - \pi)\right]$$

Since $f$ is $2\pi$-periodic, $f(\theta \pm \pi) = f(\theta + \pi)$, giving:

$$\frac{\partial^2 f}{\partial \theta^2} = \frac{1}{2}\left[f(\theta + \pi) - f(\theta)\right]$$

In [ ]:
# Higher-order gradients and the quantum Fisher information matrix

n_qubits = 2
dev_fisher = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_fisher)
def fisher_circuit(params):
    qml.RY(params[0], wires=0)
    qml.RY(params[1], wires=1)
    qml.CNOT(wires=[0, 1])
    qml.RY(params[2], wires=0)
    qml.RY(params[3], wires=1)
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))

# Compute the gradient vector and approximate Hessian
params = np.array([0.5, 0.8, 1.2, 0.3], requires_grad=True)

# Gradient
grad_fn = qml.grad(fisher_circuit)
gradient = grad_fn(params)
print(f"Parameters: {params}")
print(f"Expectation value: {fisher_circuit(params):.6f}")
print(f"Gradient: {gradient}")

# Numerically compute the Hessian using parameter shift
n_params = len(params)
hessian = np.zeros((n_params, n_params))
shift = np.pi / 2

for i in range(n_params):
    for j in range(n_params):
        pp = params.copy(); pp[i] += shift; pp[j] += shift
        pm = params.copy(); pm[i] += shift; pm[j] -= shift
        mp = params.copy(); mp[i] -= shift; mp[j] += shift
        mm = params.copy(); mm[i] -= shift; mm[j] -= shift
        hessian[i, j] = (fisher_circuit(pp) - fisher_circuit(pm) - fisher_circuit(mp) + fisher_circuit(mm)) / 4

print(f"\nHessian matrix:")
print(np.round(hessian, 4))
print(f"\nEigenvalues of Hessian: {np.round(np.linalg.eigvalsh(hessian), 4)}")

## 4. Transfer Learning with Quantum Circuits

### Concept

**Transfer learning** is a technique where a model trained on one task is adapted for a different but related task. In the quantum-classical hybrid context:

1. Use a **pre-trained classical network** (e.g., ResNet) as a feature extractor
2. Replace the final classification layer with a **variational quantum circuit**
3. Fine-tune only the quantum parameters (and optionally a few classical layers)

### Why This Works

Classical pre-trained networks learn hierarchical features:
- **Early layers**: edges, textures, colors
- **Middle layers**: shapes, patterns
- **Final layers**: task-specific features

The quantum circuit operates on the **compressed, high-level feature representation**, where:

$$\hat{y} = \text{QCircuit}\left(\boldsymbol{\theta}; \, f_{\text{pretrained}}(\mathbf{x})\right)$$

### Advantages

- **Reduces qubit requirements**: Features are already compressed to low dimensions
- **Leverages classical strengths**: Classical networks excel at feature extraction
- **Quantum enhancement**: The quantum circuit may capture complex correlations in the feature space
- **Practical on NISQ devices**: Only a small quantum circuit is needed

In [ ]:
# Transfer learning architecture demonstration
# We'll simulate the concept using a pre-trained feature extractor + quantum classifier

n_qubits = 4
n_layers = 3
dev_tl = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_tl, interface="torch")
def quantum_classifier(inputs, weights):
    """Quantum circuit used as classifier head."""
    # Encode classical features into quantum state
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)
    
    # Variational layers
    for l in range(n_layers):
        for i in range(n_qubits):
            qml.RY(weights[l, i, 0], wires=i)
            qml.RZ(weights[l, i, 1], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])
    
    return qml.expval(qml.PauliZ(0))

class HybridTransferModel(nn.Module):
    """Classical feature extractor + quantum classifier."""
    def __init__(self):
        super().__init__()
        # Simulated pre-trained feature extractor (would be ResNet etc. in practice)
        self.feature_extractor = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, n_qubits),
            nn.Tanh()  # Bound features to [-1, 1]
        )
        # Scale to [0, pi] for quantum encoding
        self.q_weights = nn.Parameter(torch.randn(n_layers, n_qubits, 2) * 0.1)
    
    def forward(self, x):
        x = x.view(-1, 784)
        features = self.feature_extractor(x)
        # Scale features for quantum encoding
        features = (features + 1) * np.pi / 2  # Map [-1,1] -> [0, pi]
        
        # Process each sample through quantum circuit
        q_out = torch.stack([quantum_classifier(f, self.q_weights) for f in features])
        return q_out

model_tl = HybridTransferModel()
print("Transfer Learning Model Architecture:")
print(f"  Classical feature extractor parameters: {sum(p.numel() for p in model_tl.feature_extractor.parameters())}")
print(f"  Quantum circuit parameters: {model_tl.q_weights.numel()}")
print(f"  Total parameters: {sum(p.numel() for p in model_tl.parameters())}")

# Test forward pass
dummy_input = torch.randn(3, 1, 28, 28)
output = model_tl(dummy_input)
print(f"\nInput shape: {dummy_input.shape}")
print(f"Output (quantum expectation values): {output.detach().numpy()}")

## 5. Quantum-Classical Neural Network Architectures

### Architecture Taxonomy

There are several ways to combine quantum and classical layers:

#### Type 1: Quantum as Final Layer ("Dressed Quantum Circuit")
$$\hat{y} = Q_{\boldsymbol{\theta}}\left(\text{ClassicalNN}(\mathbf{x})\right)$$

#### Type 2: Quantum as Middle Layer ("Quantum Sandwich")
$$\hat{y} = \text{ClassicalNN}_2\left(Q_{\boldsymbol{\theta}}\left(\text{ClassicalNN}_1(\mathbf{x})\right)\right)$$

#### Type 3: Interleaved Layers
$$\hat{y} = C_L \circ Q_L \circ C_{L-1} \circ Q_{L-1} \circ \cdots \circ C_1 \circ Q_1(\mathbf{x})$$

#### Type 4: Parallel Processing
$$\hat{y} = \text{Combine}\left(\text{ClassicalBranch}(\mathbf{x}), \, Q_{\boldsymbol{\theta}}(\mathbf{x})\right)$$

### Loss Functions for Hybrid Models

For binary classification, the quantum output $\langle Z \rangle \in [-1, 1]$ maps naturally to:

$$p(y=1|\mathbf{x}) = \frac{1 + \langle Z \rangle_{\boldsymbol{\theta}, \mathbf{x}}}{2}$$

The binary cross-entropy loss becomes:

$$\mathcal{L}(\boldsymbol{\theta}) = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log p_i + (1-y_i)\log(1-p_i)\right]$$

For multi-class problems, we use multiple quantum measurements:

$$\mathbf{p}(\mathbf{x}) = \text{softmax}\left(\langle Z_0 \rangle, \langle Z_1 \rangle, \ldots, \langle Z_{C-1} \rangle\right)$$

In [ ]:
# Implementing different hybrid architectures and comparing them

n_q = 4
dev_arch = qml.device("default.qubit", wires=n_q)

@qml.qnode(dev_arch, interface="torch")
def quantum_layer(inputs, weights):
    """Reusable quantum layer."""
    for i in range(n_q):
        qml.RY(inputs[i], wires=i)
    for i in range(n_q):
        qml.RY(weights[i, 0], wires=i)
        qml.RZ(weights[i, 1], wires=i)
    for i in range(n_q - 1):
        qml.CNOT(wires=[i, i + 1])
    qml.CNOT(wires=[n_q - 1, 0])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_q)]

# Type 1: Quantum as Final Layer
class Type1_QuantumFinal(nn.Module):
    def __init__(self):
        super().__init__()
        self.classical = nn.Sequential(
            nn.Linear(8, 16), nn.ReLU(),
            nn.Linear(16, n_q), nn.Tanh()
        )
        self.q_weights = nn.Parameter(torch.randn(n_q, 2) * 0.3)
    
    def forward(self, x):
        features = self.classical(x) * np.pi
        return torch.stack([torch.stack(quantum_layer(f, self.q_weights)) for f in features])

# Type 2: Quantum Sandwich
class Type2_QuantumSandwich(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre = nn.Sequential(nn.Linear(8, n_q), nn.Tanh())
        self.q_weights = nn.Parameter(torch.randn(n_q, 2) * 0.3)
        self.post = nn.Sequential(nn.Linear(n_q, 8), nn.ReLU(), nn.Linear(8, n_q))
    
    def forward(self, x):
        features = self.pre(x) * np.pi
        q_out = torch.stack([torch.stack(quantum_layer(f, self.q_weights)) for f in features])
        return self.post(q_out)

# Compare architectures
for name, ModelClass in [("Type 1: Quantum Final", Type1_QuantumFinal),
                          ("Type 2: Quantum Sandwich", Type2_QuantumSandwich)]:
    model = ModelClass()
    n_params = sum(p.numel() for p in model.parameters())
    x = torch.randn(2, 8)
    y = model(x)
    print(f"{name}: {n_params} params, output shape = {y.shape}")

## 6. Building and Training: Hybrid Model for MNIST Classification

Now we build a complete hybrid quantum-classical model to classify handwritten digits from the MNIST dataset. We'll focus on a binary classification task (digits 0 vs 1) to keep the quantum circuit manageable.

### Architecture

Our model follows the "dressed quantum circuit" pattern:

1. **Classical encoder**: $\mathbb{R}^{784} \to \mathbb{R}^4$ (compress 28x28 images to 4 features)
2. **Quantum variational circuit**: 4 qubits, 2 layers of $R_Y$-$R_Z$ rotations + CNOT entanglement
3. **Classical decoder**: Map $\langle Z_0 \rangle \in [-1, 1]$ to class probability

### Training Procedure

We optimize the combined loss:

$$\mathcal{L}(\boldsymbol{\phi}, \boldsymbol{\theta}) = \frac{1}{N}\sum_{i=1}^{N} \ell\left(\sigma\left(\langle Z_0 \rangle_{\boldsymbol{\phi}, \boldsymbol{\theta}, \mathbf{x}_i}\right), y_i\right)$$

where $\boldsymbol{\phi}$ are classical parameters, $\boldsymbol{\theta}$ are quantum parameters, and $\sigma$ is the sigmoid function.

In [ ]:
# Data preparation: MNIST digits 0 and 1

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Download MNIST
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Filter for digits 0 and 1
def filter_01(dataset):
    idx = (dataset.targets == 0) | (dataset.targets == 1)
    return Subset(dataset, torch.where(idx)[0])

train_01 = filter_01(train_dataset)
test_01 = filter_01(test_dataset)

# Use smaller subsets for tractable training with quantum circuits
n_train = 200
n_test = 50
train_subset = Subset(train_01, range(min(n_train, len(train_01))))
test_subset = Subset(test_01, range(min(n_test, len(test_01))))

train_loader = DataLoader(train_subset, batch_size=10, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=n_test)

print(f"Training samples: {len(train_subset)}")
print(f"Test samples: {len(test_subset)}")

# Visualize some examples
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_subset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}', fontsize=9)
    ax.axis('off')
plt.suptitle('MNIST Digits 0 and 1 (Training Samples)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Define the hybrid quantum-classical model

n_qubits_mnist = 4
n_qlayers = 2
dev_mnist = qml.device("default.qubit", wires=n_qubits_mnist)

@qml.qnode(dev_mnist, interface="torch", diff_method="backprop")
def mnist_quantum_circuit(inputs, weights):
    """
    Variational quantum circuit for MNIST classification.
    inputs: 4 features from classical encoder
    weights: shape (n_qlayers, n_qubits, 3) - RX, RY, RZ per qubit per layer
    """
    # Amplitude encoding via rotations
    for i in range(n_qubits_mnist):
        qml.Hadamard(wires=i)
        qml.RY(inputs[i], wires=i)
    
    # Variational layers
    for l in range(n_qlayers):
        # Parameterized rotations
        for i in range(n_qubits_mnist):
            qml.RX(weights[l, i, 0], wires=i)
            qml.RY(weights[l, i, 1], wires=i)
            qml.RZ(weights[l, i, 2], wires=i)
        # Entangling gates (ring topology)
        for i in range(n_qubits_mnist):
            qml.CNOT(wires=[i, (i + 1) % n_qubits_mnist])
    
    return qml.expval(qml.PauliZ(0))

class HybridMNISTModel(nn.Module):
    """Full hybrid model: classical encoder -> quantum circuit -> classifier."""
    
    def __init__(self):
        super().__init__()
        
        # Classical pre-processing (dimensionality reduction)
        self.encoder = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, n_qubits_mnist),
            nn.Tanh()  # Output in [-1, 1]
        )
        
        # Quantum circuit parameters
        self.q_weights = nn.Parameter(
            torch.randn(n_qlayers, n_qubits_mnist, 3) * 0.1
        )
        
        # Classical post-processing
        self.decoder = nn.Sequential(
            nn.Linear(1, 1)
        )
    
    def forward(self, x):
        batch_size = x.shape[0]
        x = x.view(batch_size, -1)  # Flatten
        
        # Classical encoding
        features = self.encoder(x)
        features = features * np.pi  # Scale to [-pi, pi]
        
        # Quantum processing
        q_outputs = []
        for i in range(batch_size):
            q_out = mnist_quantum_circuit(features[i], self.q_weights)
            q_outputs.append(q_out)
        q_outputs = torch.stack(q_outputs).unsqueeze(1)  # (batch, 1)
        
        # Classical post-processing
        logits = self.decoder(q_outputs).squeeze(1)
        return logits

model = HybridMNISTModel()
print("Hybrid MNIST Model:")
print(f"  Encoder parameters: {sum(p.numel() for p in model.encoder.parameters())}")
print(f"  Quantum parameters: {model.q_weights.numel()}")
print(f"  Decoder parameters: {sum(p.numel() for p in model.decoder.parameters())}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters())}")

# Print quantum circuit
print("\nQuantum circuit:")
dummy_in = torch.randn(n_qubits_mnist)
dummy_w = torch.randn(n_qlayers, n_qubits_mnist, 3)
print(qml.draw(mnist_quantum_circuit, decimals=2)(dummy_in, dummy_w))

In [ ]:
# Training loop

model = HybridMNISTModel()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

n_epochs = 15
train_losses = []
train_accs = []
test_accs = []

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in train_loader:
        batch_y = batch_y.float()
        optimizer.zero_grad()
        
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
    
    train_loss = epoch_loss / len(train_loader)
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Evaluate on test set
    model.eval()
    with torch.no_grad():
        for test_x, test_y in test_loader:
            test_out = model(test_x)
            test_preds = (torch.sigmoid(test_out) > 0.5).float()
            test_acc = (test_preds == test_y.float()).float().mean().item()
    test_accs.append(test_acc)
    
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{n_epochs} | Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:.3f} | Test Acc: {test_acc:.3f}")

In [ ]:
# Visualize training results

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(range(1, n_epochs + 1), train_losses, 'b-o', linewidth=2, markersize=5)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('BCE Loss', fontsize=12)
ax1.set_title('Training Loss', fontsize=13)
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(range(1, n_epochs + 1), train_accs, 'b-o', linewidth=2, markersize=5, label='Train')
ax2.plot(range(1, n_epochs + 1), test_accs, 'r-s', linewidth=2, markersize=5, label='Test')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Classification Accuracy', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0.4, 1.05])

plt.suptitle('Hybrid Quantum-Classical Model Training on MNIST (0 vs 1)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nFinal Train Accuracy: {train_accs[-1]:.3f}")
print(f"Final Test Accuracy: {test_accs[-1]:.3f}")

In [ ]:
# Visualize the quantum layer's learned representations

model.eval()

# Extract features from the classical encoder
all_features = []
all_labels = []

with torch.no_grad():
    for x, y in DataLoader(train_subset, batch_size=50):
        x_flat = x.view(-1, 784)
        features = model.encoder(x_flat)
        all_features.append(features.numpy())
        all_labels.append(y.numpy())

all_features = np.concatenate(all_features)
all_labels = np.concatenate(all_labels)

# Plot the 4D features reduced to 2D via PCA
pca = PCA(n_components=2)
features_2d = pca.fit_transform(all_features)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for label, color, marker in [(0, 'blue', 'o'), (1, 'red', 's')]:
    mask = all_labels == label
    ax1.scatter(features_2d[mask, 0], features_2d[mask, 1], 
                c=color, marker=marker, alpha=0.6, label=f'Digit {label}', s=50)
ax1.set_xlabel('PC1', fontsize=12)
ax1.set_ylabel('PC2', fontsize=12)
ax1.set_title('Classical Encoder Output (PCA)', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Show the distribution of quantum circuit outputs
q_outputs = []
with torch.no_grad():
    for x, y in DataLoader(train_subset, batch_size=50):
        out = model(x)
        q_outputs.append(out.numpy())

q_outputs = np.concatenate(q_outputs)

for label, color in [(0, 'blue'), (1, 'red')]:
    mask = all_labels == label
    ax2.hist(q_outputs[mask], bins=20, alpha=0.5, color=color, label=f'Digit {label}', density=True)
ax2.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax2.set_xlabel('Model Output (Logit)', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Hybrid Model Output Distribution', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Analysis: Classical vs Hybrid Performance

Let us compare the hybrid model against a purely classical model with a similar parameter count.

In [ ]:
# Classical baseline model with similar parameter count

class ClassicalBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.net(x.view(-1, 784)).squeeze(1)

classical_model = ClassicalBaseline()
classical_optimizer = optim.Adam(classical_model.parameters(), lr=0.005)

classical_train_losses = []
classical_train_accs = []
classical_test_accs = []

for epoch in range(n_epochs):
    classical_model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in train_loader:
        batch_y = batch_y.float()
        classical_optimizer.zero_grad()
        outputs = classical_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        classical_optimizer.step()
        epoch_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
    
    classical_train_losses.append(epoch_loss / len(train_loader))
    classical_train_accs.append(correct / total)
    
    classical_model.eval()
    with torch.no_grad():
        for test_x, test_y in test_loader:
            test_out = classical_model(test_x)
            test_preds = (torch.sigmoid(test_out) > 0.5).float()
            ctest_acc = (test_preds == test_y.float()).float().mean().item()
    classical_test_accs.append(ctest_acc)

# Comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, n_epochs+1), train_losses, 'b-o', label='Hybrid (QC)', markersize=4)
ax1.plot(range(1, n_epochs+1), classical_train_losses, 'g-s', label='Classical', markersize=4)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss Comparison'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(range(1, n_epochs+1), test_accs, 'b-o', label='Hybrid (QC)', markersize=4)
ax2.plot(range(1, n_epochs+1), classical_test_accs, 'g-s', label='Classical', markersize=4)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Test Accuracy')
ax2.set_title('Test Accuracy Comparison'); ax2.legend(); ax2.grid(True, alpha=0.3)
ax2.set_ylim([0.4, 1.05])

plt.suptitle('Hybrid Quantum-Classical vs Pure Classical Model', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Hybrid model params: {sum(p.numel() for p in model.parameters())}")
print(f"Classical model params: {sum(p.numel() for p in classical_model.parameters())}")
print(f"\nHybrid final test acc: {test_accs[-1]:.3f}")
print(f"Classical final test acc: {classical_test_accs[-1]:.3f}")

## 8. Summary and Key Takeaways

### What We Learned

1. **Hybrid quantum-classical models** combine the strengths of both paradigms: classical networks handle high-dimensional data processing while quantum circuits provide expressive parameterized transformations.

2. **The parameter shift rule** enables exact gradient computation for quantum circuits:
$$\frac{\partial f}{\partial \theta} = \frac{f(\theta + \pi/2) - f(\theta - \pi/2)}{2}$$

3. **Transfer learning** with quantum circuits is practical: use pre-trained classical networks as feature extractors and quantum circuits as classifiers.

4. **Architecture choices** matter: the "dressed quantum circuit" (classical-quantum-classical) is the most common and practical pattern.

### Challenges

- **Barren plateaus**: Deep variational circuits may have vanishing gradients
- **Training speed**: Quantum circuit evaluation is slow compared to classical layers
- **Noise**: Real quantum hardware adds noise to all computations
- **Limited qubits**: Current devices restrict the quantum layer size

### Next Steps

In the next notebook, we'll explore **Quantum Neural Networks (QNNs)** in depth, including quantum convolutional neural networks and the data re-uploading classifier.

---

### References

1. Mitarai, K., et al. "Quantum circuit learning." *Physical Review A* 98.3 (2018): 032309.
2. Schuld, M., et al. "Evaluating analytic gradients on quantum hardware." *Physical Review A* 99.3 (2019): 032331.
3. Mari, A., et al. "Transfer learning in hybrid classical-quantum neural networks." *Quantum* 4 (2020): 340.
4. Sim, S., et al. "Expressibility and entangling capability of parameterized quantum circuits." *Advanced Quantum Technologies* 2.12 (2019): 1900070.